In [9]:
import pandas as pd
from catboost import CatBoostClassifier

#### Load the train data and drop irrelevant features


In [10]:
df = pd.read_csv('train.csv',sep=',')
y = df["Survived"]
df = df.drop(columns=["PassengerId","Name","Survived","Cabin","Ticket"],axis=1)

- There are some missing values in the Age column, so I decided to fill them with the mean age of each Pclass group.


In [11]:
print(df.isna().sum())
df_invalid_age=df[df['Age'].isna()] ##create a df for the data with NaN age
print(df_invalid_age["Pclass"].value_counts())##checking which Pclass has the most NaN age

mean_ages = df.groupby("Pclass")["Age"].mean()

## Filling the Nan ages
df["Age"] = df.apply(
    lambda row: mean_ages[row["Pclass"]] if pd.isna(row["Age"]) else row["Age"], axis=1
)


Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64
Pclass
3    136
1     30
2     11
Name: count, dtype: int64


There are only 2 missing data points, so I decided to examine each individually. and then fill based on the mode of their Pclass

In [12]:
print(df[df["Embarked"].isna()]) #it seems that both missing data are Pclass = 1
print(df[df["Pclass"]==1]["Embarked"].value_counts()) ## checking the most common embarked value for first class
df["Embarked"] = df["Embarked"].fillna(df[df["Pclass"]==1]["Embarked"].mode()[0]) # fill with the Pclass==1 mode
X_train = df

     Pclass     Sex   Age  SibSp  Parch  Fare Embarked
61        1  female  38.0      0      0  80.0      NaN
829       1  female  62.0      0      0  80.0      NaN
Embarked
S    127
C     85
Q      2
Name: count, dtype: int64


### Load the test dataset and adjust it to match the shape of the train dataset.

In [13]:
df_test = pd.read_csv('test.csv',sep=',')
passenger_id = df_test["PassengerId"]
df_test = df_test.drop(['Name','Cabin','PassengerId',"Ticket"],axis=1)

### Applying the same changes


In [14]:
print(df_test.isna().sum())
df_test_invalid_age=df_test[df_test['Age'].isna()]
print(df_test_invalid_age["Pclass"].value_counts())

mean_ages = df_test.groupby("Pclass")["Age"].mean()

df_test["Age"] = df_test.apply(
    lambda row: mean_ages[row["Pclass"]] if pd.isna(row["Age"]) else row["Age"], axis=1
)

print(df_test[df_test["Embarked"].isna()])
print(df_test[df_test["Pclass"]==1]["Embarked"].value_counts())
df_test["Embarked"] = df_test["Embarked"].fillna(df_test[df_test["Pclass"]==1]["Embarked"].mode()[0])
X_test = df_test

Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64
Pclass
3    72
1     9
2     5
Name: count, dtype: int64
Empty DataFrame
Columns: [Pclass, Sex, Age, SibSp, Parch, Fare, Embarked]
Index: []
Embarked
C    56
S    50
Q     1
Name: count, dtype: int64


Choose the Model, train and predict

In [15]:
model = CatBoostClassifier(iterations= 1000,learning_rate=0.1,cat_features=["Embarked","Sex"],task_type='GPU',loss_function="Logloss",verbose=100)
model.fit(X_train, y)
y_pred = model.predict(X_test)
submission = pd.DataFrame({
   'PassengerId': passenger_id,  
  'Survived': y_pred                 
})
submission.to_csv('submission.csv',index=False)

0:	learn: 0.6265653	total: 26ms	remaining: 26s
100:	learn: 0.3128395	total: 2.03s	remaining: 18.1s
200:	learn: 0.2798555	total: 4.15s	remaining: 16.5s
300:	learn: 0.2499426	total: 6.4s	remaining: 14.9s
400:	learn: 0.2276473	total: 8.71s	remaining: 13s
500:	learn: 0.2135926	total: 11s	remaining: 10.9s
600:	learn: 0.2002640	total: 13.3s	remaining: 8.81s
700:	learn: 0.1899113	total: 15.6s	remaining: 6.64s
800:	learn: 0.1781653	total: 18s	remaining: 4.47s
900:	learn: 0.1677628	total: 20.4s	remaining: 2.24s
999:	learn: 0.1596050	total: 22.7s	remaining: 0us


In [16]:
print(f"\n \nImportance for each feature:\n\n{model.get_feature_importance(prettified=True)}")


 
Importance for each feature:

  Feature Id  Importances
0        Age    26.506366
1       Fare    25.888080
2        Sex    21.906258
3   Embarked     7.780463
4      SibSp     7.534487
5     Pclass     6.652084
6      Parch     3.732263
